### 將jupter 處理成可以讀取資料夾下的其他模組

In [ ]:
import sys
import os
import pandas as pd
# notebook 所在目錄（通常 cwd = rebuild_data）加入 sys.path
_here = os.path.abspath(os.getcwd())
if _here not in sys.path:
    sys.path.insert(0, _here)


### 引入通用的處理模塊

In [ ]:
from parse import read_tej_txt, one_txt, read_tej_folder , txt_to_parquet
import polars as pl

In [ ]:
rebuild_root = "/mnt/c/Users/Ultra7/TEJ_data/rebuild_20150601~20260531" 
save_path = "/mnt/c/Users/Ultra7/TEJ_data/data_daily"

In [ ]:
import importlib

def reload_modules(module_list):
    """重新載入指定的模組列表"""
    for mod in module_list:
        importlib.reload(mod)

## 資料 import 
### 引入日歷 
處理成 只有日期 並修正格式

In [ ]:
import Calendar

calendar_path = os.path.join(rebuild_root, "calendar", "calendar.txt")
if not os.path.exists(calendar_path):
    raise FileNotFoundError(f"檔案不存在：{calendar_path}，請確認路徑是否正確。")
calendar_df = read_tej_txt(calendar_path, Calendar.CALENDAR_SCHEMA).select(Calendar.keep_col)
calendar_df.head()


### 引入停止交易的日曆

In [ ]:
import Trading_Halt

stop_trading_path = os.path.join(rebuild_root, "trading_halt")
trading_halt_df = read_tej_txt(
    one_txt(stop_trading_path),
    Trading_Halt.TRADING_HALT_SCHEMA,
).select(Trading_Halt.keep_col)

# Y9999 為加權指數
# 完全相等 / 部分包含 / 手工 (symbol, 年月日YYYYMMDD)（含任一即 avoid_early=1）
AVOID_EARLY_REASONS: list[str] = ["減資", "現金減資", "違規財報", "自行申請"]
AVOID_EARLY_REASONS_PARTIAL: list[str] = ["面額變更"]
AVOID_EARLY_PAIRS: list[tuple[str, int]] = [
    ("4413", 20180515),  # 前面被打入全額交割 未改善 視為可先預知
    ("2833", 20151002),  # 台灣人壽 併入中國金控 有提前公告
    ("2823", 20211220),  # 中國人壽 併入開發金
]
_exact = pl.col("reason").is_in(AVOID_EARLY_REASONS)
_partial = (
    pl.any_horizontal(
        *[pl.col("reason").str.contains(p, literal=True) for p in AVOID_EARLY_REASONS_PARTIAL]
    )
    if AVOID_EARLY_REASONS_PARTIAL
    else pl.lit(False)
)
_pair_df = (
    pl.DataFrame(AVOID_EARLY_PAIRS, schema={"symbol": pl.Utf8, "_ymd": pl.Int64}, orient="row")
    if AVOID_EARLY_PAIRS
    else pl.DataFrame(schema={"symbol": pl.Utf8, "_ymd": pl.Int64})
).with_columns(pl.lit(True).alias("_manual"))
trading_halt_df = (
    trading_halt_df
    .with_columns(
        pl.col("start_halt").dt.strftime("%Y%m%d").cast(pl.Int64).alias("_ymd")
    )
    .join(_pair_df, on=["symbol", "_ymd"], how="left")
    .with_columns(
        (_exact | _partial | pl.col("_manual").fill_null(False))
        .cast(pl.Int8)
        .alias("avoid_early")
    )
    .drop(["_ymd", "_manual"])
)
trading_halt_df.head()


In [ ]:
# schema 已將 start_halt / resume_trading_date 轉成 Date；此格保留作型別確認
assert trading_halt_df["start_halt"].dtype == pl.Date
assert trading_halt_df["resume_trading_date"].dtype == pl.Date
print("trading_halt 日期欄型別 OK")


In [ ]:
# EDA：恢復交易日缺值
resume_empty = pl.col("resume_trading_date").is_null()

print("=== 總覽 ===")
print(f"總筆數: {trading_halt_df.height}")
print(trading_halt_df.select(
    pl.col("start_halt").is_not_null().sum().alias("有開始日"),
    resume_empty.sum().alias("無恢復日"),
    (~resume_empty).sum().alias("有恢復日"),
))

no_resume = trading_halt_df.filter(
    pl.col("start_halt").is_not_null() & pl.col("resume_trading_date").is_null()
)
print(f"\n=== 有開始日、無恢復日：{no_resume.height} 筆 ===")
print("原因分布:")
print(no_resume.group_by("reason").len().sort("len", descending=True))
print(f"\n=== symbol（共 {no_resume['symbol'].n_unique()} 檔）===")
print(no_resume.group_by("symbol").len().sort("len", descending=True))
print("\nsample:")
no_resume.head(20)


In [ ]:
# 理論上 halt 中 symbol "Y9999" 的日期不會出現在 calendar 中
y9999_halt_dates = trading_halt_df.filter(pl.col("symbol") == "Y9999")["start_halt"]
calendar_dates = calendar_df["date"].unique()
y9999_in_calendar = y9999_halt_dates.filter(y9999_halt_dates.is_in(calendar_dates.to_list()))

if len(y9999_in_calendar) == 0:
    print("理論一致：Y9999 halt 年月日無出現在 calendar 中")
else:
    print("以下 Y9999 halt 年月日 竟出現在 calendar 中，請檢查！")
    print(y9999_in_calendar)


### 處理OHLCV

In [ ]:
from OHLCV import OHLCV_SCHEMA , process_OHLCV_df 
import OHLCV

In [ ]:
OHLCV_path = os.path.join(rebuild_root, "Daily_Unadjusted_Price")
OHLCV_df = read_tej_folder(OHLCV_path, OHLCV_SCHEMA, allow_market_split=True)
OHLCV_df.select(OHLCV.keep_col)
OHLCV_df = process_OHLCV_df(OHLCV_df)
OHLCV_df.head()

In [ ]:
# 找出 OHLCV_df 內有哪些 date 不在 calendar_df 裡
# 先確保 calendar_df 和 OHLCV_df 的 date 欄位型態一致
# 若 calendar_df 只有一欄 'date' 且型別為 date/datetime

ohlcv_dates = OHLCV_df['date'].unique()
calendar_dates = calendar_df['date'].unique()

missing_dates = ohlcv_dates.filter(~ohlcv_dates.is_in(calendar_dates.to_list()))

if len(missing_dates) > 0 :
    print(missing_dates)
else:
    print("OHLCV_df 內所有日期都在 calendar_df 裡")

In [ ]:
# 全額交割(Y) 或非 TSE/OTC → untradable=1；其餘明確為 0（避免 Kleene null）
OHLCV_df = OHLCV_df.with_columns(
    (
        (pl.col("full_delivery") == 1).fill_null(False)
        | (~pl.col("market").is_in(["TSE", "OTC"])).fill_null(False)
    )
    .cast(pl.Int8)
    .alias("untradable")
)

### 處理暫停交易的日期
將暫停交易表的時間區段填入 OHLCV的欄位並將其標記為不可交易

In [ ]:
trading_halt_df.filter(pl.col('symbol') == '8932')

In [ ]:
# 停牌區間填入 OHLCV：[start_halt, resume)；另標 start 前一交易日 untradable
# avoid_early 只寫入「start_halt 前一交易日」（可提前避開的停牌）
halts = (
    trading_halt_df
    .select(["symbol", "start_halt", "resume_trading_date", "reason", "avoid_early"])
    .drop_nulls("start_halt")
    .join(OHLCV_df.select("symbol").unique(), on="symbol", how="inner")
)

halted = (
    OHLCV_df.select(["symbol", "date"])
    .join(halts, on="symbol", how="inner")
    .filter(
        (pl.col("date") >= pl.col("start_halt"))
        & (
            pl.col("resume_trading_date").is_null()
            | (pl.col("date") < pl.col("resume_trading_date"))
        )
    )
    .select(["symbol", "date"])
    .unique()
)

# start 前一交易日 + 該停牌事件的 avoid_early
pre_halt = (
    halts.select(["symbol", "start_halt", "avoid_early"])
    .join(calendar_df.select("date"), how="cross")
    .filter(pl.col("date") < pl.col("start_halt"))
    .group_by(["symbol", "start_halt"])
    .agg(
        pl.col("date").max().alias("date"),
        pl.col("avoid_early").first().alias("avoid_early"),
    )
    .drop_nulls("date")
)
# 同一 (symbol, date) 若對應多段停牌，取 max(avoid_early)
pre_halt_flag = (
    pre_halt
    .group_by(["symbol", "date"])
    .agg(pl.col("avoid_early").max().cast(pl.Int8).alias("avoid_early"))
)
pre_halt_days = pre_halt_flag.select(["symbol", "date"])

halted = (
    pl.concat([halted, pre_halt_days])
    .unique()
    .with_columns(pl.lit(1).cast(pl.Int8).alias("halt_flag"))
)

OHLCV_df = (
    OHLCV_df
    .join(halted, on=["symbol", "date"], how="left")
    .join(pre_halt_flag, on=["symbol", "date"], how="left")
    .with_columns(
        pl.max_horizontal(pl.col("untradable"), pl.col("halt_flag").fill_null(0))
        .cast(pl.Int8)
        .alias("untradable"),
        pl.col("avoid_early").fill_null(0).cast(pl.Int8).alias("avoid_early"),
    )
    .drop("halt_flag")
)

print("halted rows:", halted.height, "(含 start 前一交易日)")
print(
    "avoid_early=1 列數:",
    OHLCV_df.filter(pl.col("avoid_early") == 1).height,
    "（應落在 start 前一交易日）",
)
OHLCV_df.filter(pl.col("avoid_early") == 1).head()


In [ ]:
# 處理過的暫停交易資料的原因統計
# halted 無 reason，以 halts 為主；只保留 start_halt 落在 halted 且 avoid_early==0 的事件
reason_stats = (
    halts
    .filter(pl.col("avoid_early") == 0)
    .join(
        halted.select(["symbol", pl.col("date").alias("start_halt")]).unique(),
        on=["symbol", "start_halt"],
        how="semi",
    )
    .group_by("reason")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
# 不統計內容包含 "因有重大訊息待公佈" 的數據
# 這些內容都是無法提前知道的
exclude_keywords = ["重大訊息待公佈", "重大訊息待公布", "重大事項待公布", "重大訊息", "重大事項", "重大消息"]
reason_stats = reason_stats.filter(~pl.col("reason").str.contains('|'.join(exclude_keywords)))

print("各暫停交易理由的統計：")
# 計算 reason_stats 的 count 欄位和
total_count = reason_stats["count"].sum()
print("count 欄位總和:", total_count)

# 列出剩餘理由的原始資料
filtered_reasons = reason_stats["reason"].to_list()
filtered_halts = halts.filter(
    (pl.col("avoid_early") == 0) &
    (pl.col("reason").is_in(filtered_reasons))
).select(["symbol", "start_halt", "resume_trading_date", "reason"])
print("全部過濾後 halt 資料：")
#print(filtered_halts)

# 【新增】指定條件的篩選：指定標的(specific_symbols)與指定 start_halt 日期(specific_start_halts)
# 請自行指定你要查的範例

# 缺少指定的key word 無法判定 手工處理
# 4746 20220809 公布訊息
# 6838 20220809 公布訊息
# 7772 20260420 漲84% 熔斷
# 5225 20210315 公布訊息
# 6224 20200708 公布訊息
# 6885 20231222 跌60% 
# 6589 20210323 公布訊息
# 2102 20230210 公布訊息
# 4414 20180226 公布訊息
# 4743 20200415 公布訊息
# 2474 20200818 公布訊息
# 6438 20210415 公布訊息
# 5432 20191001 公布訊息
specific_symbols = [
    "4746", "7772" , "5225" , "6838" , "6224" , "6885" , "6589" , "2102" , "4414" , "4743" , "2474" , "6438" , "5432"
    ]  # 這裡可替換為你要查的標的代碼
specific_start_halts = [
    "2022-08-09" , "2026-04-20" , "2021-03-15" , "2022-08-09" , "2020-07-08" , "2023-12-22" , "2021-03-23" , "2023-02-10" , 
    "2018-02-26" , "2020-04-15" , "2020-08-18" , "2021-04-15" , "2019-10-01"
    ]  # 替換為你要查的 start_halt 日期(字串格式、可多個)
filtered_specific = filtered_halts.filter(
    ~(
        pl.col("symbol").is_in(specific_symbols) &
        pl.col("start_halt").cast(pl.Utf8).is_in(specific_start_halts)
    )
)

print("指定標的、指定start_halt的篩選結果：")

filtered_specific

In [ ]:
# 驗證：
# 1) [start_halt, resume) 皆 untradable==1
# 2) start 前一交易日亦 untradable==1
# 3) 恢復日可不在「自己的」暫停區間；若同日又是下一段 start（或下一段的 start 前一日），仍可暫停
expected_halted = (
    OHLCV_df.select(["symbol", "date", "untradable"])
    .join(halts, on="symbol", how="inner")
    .filter(
        (pl.col("date") >= pl.col("start_halt"))
        & (
            pl.col("resume_trading_date").is_null()
            | (pl.col("date") < pl.col("resume_trading_date"))
        )
    )
)
missing = expected_halted.filter(pl.col("untradable") != 1)

expected_pre_halt = (
    halts.select(["symbol", "start_halt"])
    .join(calendar_df.select("date"), how="cross")
    .filter(pl.col("date") < pl.col("start_halt"))
    .group_by(["symbol", "start_halt"])
    .agg(pl.col("date").max().alias("date"))
    .select(["symbol", "date"])
    .drop_nulls("date")
    .unique()
    .join(OHLCV_df.select(["symbol", "date", "untradable"]), on=["symbol", "date"], how="inner")
)
missing_pre = expected_pre_halt.filter(pl.col("untradable") != 1)

# 恢復日落在 expected_halted：多半是「恢復日 = 下一段 start」（銜接停牌），屬正確
resume_days = (
    halts.filter(pl.col("resume_trading_date").is_not_null())
    .select(["symbol", pl.col("resume_trading_date").alias("date")])
    .unique()
)
start_days = halts.select(["symbol", pl.col("start_halt").alias("date")]).unique()
pre_halt_days = expected_pre_halt.select(["symbol", "date"]).unique()
# 恢復日若等於下一段 start，或等於某段的 start 前一日，都算合理銜接
allowed_resume_halt = pl.concat([start_days, pre_halt_days]).unique()

resume_in_halted = (
    OHLCV_df.select(["symbol", "date"])
    .join(resume_days, on=["symbol", "date"], how="inner")
    .join(expected_halted.select(["symbol", "date"]).unique(), on=["symbol", "date"], how="inner")
)
resume_wrongly_in_halt = resume_in_halted.join(allowed_resume_halt, on=["symbol", "date"], how="anti")
chained_ok = resume_in_halted.join(allowed_resume_halt, on=["symbol", "date"], how="inner")

print("應暫停但未標記:", missing.height)
print("start前一日應暫停但未標記:", missing_pre.height)
print("恢復日銜接下一段/其前一日(正確仍暫停):", chained_ok.height)
print("恢復日被誤算進暫停區間:", resume_wrongly_in_halt.height)
print("暫停區間覆蓋 OHLCV 筆數:", expected_halted.select(["symbol", "date"]).unique().height)
print("start前一日覆蓋 OHLCV 筆數:", expected_pre_halt.height)

if missing.height == 0 and missing_pre.height == 0 and resume_wrongly_in_halt.height == 0:
    print("ok: 暫停區間與 start 前一日條件滿足")
else:
    if missing.height:
        print("missing:")
        display(missing.head())
    if missing_pre.height:
        print("missing_pre:")
        display(missing_pre.head())
    if resume_wrongly_in_halt.height:
        print("resume_wrongly_in_halt:")
        display(resume_wrongly_in_halt.head())
if chained_ok.height:
    print("銜接樣本:")
    display(chained_ok)

## 處理復權因子
先標記價格跳空日期 並先處理現金除息

### 引入現金除息資料

In [ ]:
import Cash_Dividends

cash_dividend_path = os.path.join(rebuild_root, "Cash_Dividends")
cash_dividend_df = read_tej_txt(
    one_txt(cash_dividend_path),
    Cash_Dividends.CASH_DIVIDENDS_SCHEMA,
).select(Cash_Dividends.keep_col)
cash_dividend_df.head()


#### 計算除息的復權因子

In [ ]:
# date=除息日；adj_factor = prev_close / ex_div_ref_price（分母須 > 0）
# 分母 ≤0 或缺價 → adj_factor=null，並打印出來
_ok = (
    pl.col("prev_close").is_not_null()
    & (pl.col("prev_close") > 0)
    & pl.col("ex_div_ref_price").is_not_null()
    & (pl.col("ex_div_ref_price") > 0)
)
dividend_info = cash_dividend_df.with_columns(
    pl.when(_ok)
    .then(pl.col("prev_close") / pl.col("ex_div_ref_price"))
    .otherwise(None)
    .alias("adj_factor")
)
_bad = dividend_info.filter(pl.col("adj_factor").is_null())
if _bad.height:
    print(f"cash dividend adj_factor null：{_bad.height} 筆")
    print(_bad.select(["symbol", "date", "dividend_per_share", "prev_close", "ex_div_ref_price"]))
else:
    print("cash dividend adj_factor：無 null")
dividend_info.head()


In [ ]:
# 從OHLCV中的symbol欄位抓出所有unique的，以此為標準，篩除dividend_info中的symbol
valid_symbols = set(OHLCV_df["symbol"].unique())
dividend_info = dividend_info.filter(pl.col("symbol").is_in(valid_symbols))
dividend_info.head()

#### NA 排查
原因是某些股票在不可交易的時間 依然有除息 所以會有紀錄 但是沒有開收盤價  
排查方法是將symbol,date形成一個pair去比較ohlcv 內有沒有這組

In [ ]:
# 找出 dividend_info 中任意欄位有 NA 的 row，並確認是否存在於 OHLCV
na_rows = (
    dividend_info
    .filter(pl.any_horizontal(pl.col(c).is_null() for c in dividend_info.columns))
    .select(["symbol", "date"])
    .unique()
)
print("na_rows:", na_rows.height)
print(na_rows.head())

if na_rows.height == 0:
    has_pair_df = pl.DataFrame(
        schema={"symbol": pl.Utf8, "date": pl.Date, "exists_in_ohlcv": pl.Boolean}
    )
else:
    has_pair_df = (
        na_rows.join(
            OHLCV_df.select(["symbol", "date"]).unique().with_columns(pl.lit(True).alias("exists_in_ohlcv")),
            on=["symbol", "date"],
            how="left",
        )
        .with_columns(pl.col("exists_in_ohlcv").fill_null(False))
    )

print("check_na_pairs (只列出存在於 OHLCV 的)：")
print(has_pair_df.filter(pl.col("exists_in_ohlcv")))


In [ ]:
# 不存在於 OHLCV 的 (symbol, date) 直接 drop（停牌日仍有除息紀錄但無價）
no_pair = has_pair_df.filter(~pl.col("exists_in_ohlcv")).select(["symbol", "date"])
dividend_info = dividend_info.join(no_pair, on=["symbol", "date"], how="anti")

print("dividend_info after dropping rows not present in ohlcv:")
print(dividend_info.head())

still_na_rows = dividend_info.filter(
    pl.any_horizontal(pl.col(c).is_null() for c in dividend_info.columns)
)
if still_na_rows.height > 0:
    print("仍有任意欄位為 NA 的 rows：")
    print(still_na_rows.select(["symbol", "date"]).unique())
else:
    print("drop後，dividend_info 中已無任一欄位為 NA 的 row")


In [ ]:
# 在 OHLCV_df 新增 adj_factor，預設 1；除息日覆寫為 dividend 的 adj_factor
# 已知同日多筆配息（息值不同但 prev_close / ex_div_ref_price 相同）→ 指定後只留一筆
from datetime import date as _date

CASH_DIV_DUP_OK_PAIRS: list[tuple[str, _date]] = [
    ("2236", _date(2021, 9, 13)), # 同時發兩筆 ky公司 延遲
    ("2505", _date(2020, 8, 18)), # 季配息 + 土地出售
    ("2505", _date(2021, 11, 24)), # 兩季一起發
    ("2542", _date(2019, 10, 2)), # 2018 全年 + 2019改成季配
    ("2756", _date(2021, 8, 23)),  # 年配 + 季配
    ("4912", _date(2021, 7, 8)), # 20年底季配 + 21年初季配
]


OHLCV_df = OHLCV_df.with_columns(pl.lit(1.0).alias("adj_factor"))

_dup = (
    dividend_info.filter(pl.col("adj_factor").is_not_null())
    .group_by(["symbol", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .sort(["symbol", "date"])
)
if _dup.height:
    _ok_pair_df = pl.DataFrame(
        CASH_DIV_DUP_OK_PAIRS,
        schema={"symbol": pl.Utf8, "date": pl.Date},
        orient="row",
    ) if CASH_DIV_DUP_OK_PAIRS else pl.DataFrame(
        schema={"symbol": pl.Utf8, "date": pl.Date}
    )
    _allowed = _dup.join(_ok_pair_df, on=["symbol", "date"], how="semi")
    _not_allowed = _dup.join(_ok_pair_df, on=["symbol", "date"], how="anti")

    if _not_allowed.height:
        _show = dividend_info.join(
            _not_allowed.select(["symbol", "date"]), on=["symbol", "date"], how="inner"
        ).sort(["symbol", "date"])
        print("未在 CASH_DIV_DUP_OK_PAIRS 的重複（需人工處理）：")
        display(_not_allowed)
        display(_show)
        raise ValueError(
            f"cash dividend 有 {_not_allowed.height} 組未允許的重複 (symbol, date)"
        )

    _dup_rows = dividend_info.join(
        _allowed.select(["symbol", "date"]), on=["symbol", "date"], how="inner"
    )
    _price_bad = (
        _dup_rows.group_by(["symbol", "date"])
        .agg(
            pl.col("prev_close").n_unique().alias("n_prev"),
            pl.col("ex_div_ref_price").n_unique().alias("n_ref"),
            pl.col("adj_factor").n_unique().alias("n_adj"),
        )
        .filter(
            (pl.col("n_prev") > 1)
            | (pl.col("n_ref") > 1)
            | (pl.col("n_adj") > 1)
        )
    )
    if _price_bad.height:
        print("允許的重複組但 prev_close / ex_div_ref_price / adj_factor 不一致：")
        display(_price_bad)
        display(
            _dup_rows.join(_price_bad.select(["symbol", "date"]), on=["symbol", "date"], how="inner")
            .sort(["symbol", "date"])
        )
        raise ValueError("同日重複配息的價格欄不一致，無法自動去重")

    print(
        f"允許的同日重複配息 {_allowed.height} 組："
        f"prev_close/ex_div_ref_price 相同 → keep first"
    )
    display(_allowed)
    dividend_info = dividend_info.unique(subset=["symbol", "date"], keep="first")

merge_df = dividend_info.filter(pl.col("adj_factor").is_not_null()).select(
    ["symbol", "date", "adj_factor"]
)
_dup2 = merge_df.group_by(["symbol", "date"]).len().filter(pl.col("len") > 1)
if _dup2.height:
    raise ValueError(f"去重後仍有重複：{_dup2.height} 組")

OHLCV_df = (
    OHLCV_df.join(merge_df, on=["symbol", "date"], how="left")
    .with_columns(
        pl.coalesce(pl.col("adj_factor_right"), pl.col("adj_factor")).alias("adj_factor")
    )
    .drop("adj_factor_right")
)

print("OHLCV_df after merging adj_factor:")
OHLCV_df.head()





### 引入 權 改變的數據
可能會有 除權 / 減資 等等的狀況

In [ ]:
# 建立除權的 file 路徑
from Corporate_Actions import CORPORATE_ACTIONS_SCHEMA
import Corporate_Actions
ex_rights_file_path = os.path.join(rebuild_root, "Corporate_Actions")
ex_rights_df = read_tej_txt(
    one_txt(ex_rights_file_path),
    CORPORATE_ACTIONS_SCHEMA,
)
ex_rights_df.head()


In [ ]:
# 只保留 OHLCV 有 (symbol, date) 的除權列（無價日無法掛 adj_factor）
# 私募不做公開除權參考價調整 → 排除；配股（盈餘/公積增資）要留，交給下一格用數量欄篩
ex_rights_df = ex_rights_df.join(
    OHLCV_df.select(["symbol", "date"]).unique(),
    on=["symbol", "date"],
    how="semi",
)
ex_rights_df = ex_rights_df.filter(
    pl.col("private_placement_yn").fill_null("N") != "Y"
)


#### 配股（盈餘／公積增資）


In [ ]:
# 除權復權：只取盈餘增資 / 公積增資（配股，證交所有除權參考價）
# 現金增資（現增認購）不走 prev_close/ex_rights_ref；私募已在上一格排除
# 保留 ref_price 供後續與現金除息同日比對
has_stock_dividend_increase = (
    (
        pl.col("retained_earnings_increase_thousand").is_not_null()
        & (pl.col("retained_earnings_increase_thousand") != 0)
    )
    | (
        pl.col("capital_reserve_increase_thousand").is_not_null()
        & (pl.col("capital_reserve_increase_thousand") != 0)
    )
)
has_prices = (
    pl.col("ex_rights_ref_price").is_not_null()
    & (pl.col("ex_rights_ref_price") > 0)
    & pl.col("prev_close").is_not_null()
    & (pl.col("prev_close") > 0)
)

ex_rights_adj_factor_df = (
    ex_rights_df
    .filter(has_stock_dividend_increase & has_prices)
    .select([
        "symbol",
        "date",
        (pl.col("prev_close") / pl.col("ex_rights_ref_price")).alias("adj_factor"),
        pl.col("ex_rights_ref_price").alias("ref_price"),
    ])
    .sort(["symbol", "date"])
)

_dup = (
    ex_rights_adj_factor_df.group_by(["symbol", "date"]).len().filter(pl.col("len") > 1)
)
if _dup.height:
    display(_dup)
    display(ex_rights_adj_factor_df.join(_dup.select(["symbol", "date"]), on=["symbol", "date"], how="inner"))
    raise ValueError(f"ex_rights_adj_factor_df 有 {_dup.height} 組重複 (symbol, date)")
print("ex_rights_adj_factor_df（配股）唯一 OK，筆數:", ex_rights_adj_factor_df.height)
ex_rights_adj_factor_df.head()


In [ ]:
# 將 ex_rights_adj_factor_df 篩出的 (symbol, date) pair 從 ex_rights_df 過濾掉
exclude_pairs = ex_rights_adj_factor_df.select(["symbol", "date"])
ex_rights_df = ex_rights_df.join(exclude_pairs, on=["symbol", "date"], how="anti")
ex_rights_df

#### 減資

In [ ]:
# 減資列：觀察價格齊全比例；無資料時不除法
has_capital_reduction = ex_rights_df.filter(
    (pl.col("capital_reduction_thousand").is_not_null())
    & (pl.col("capital_reduction_thousand") > 0)
)

total_rows = has_capital_reduction.height
has_both_prices = has_capital_reduction.filter(
    pl.col("ex_rights_ref_price").is_not_null() & pl.col("prev_close").is_not_null()
)
count_has_both_prices = has_both_prices.height

print(f"capital_reduction_thousand 有值 (非空且大於0) 的資料筆數: {total_rows}")
print(f"其中 ex_rights_ref_price 和 prev_close 皆不為空的筆數: {count_has_both_prices}")
if total_rows == 0:
    print("無減資資料，跳過比例計算")
else:
    print(f"比例: {count_has_both_prices / total_rows:.2%}")


In [ ]:
# 減資事件落在停牌區間 [start_halt, resume) 者：附上起迄日 + adj_factor
_ok = (
    pl.col("prev_close").is_not_null()
    & (pl.col("prev_close") > 0)
    & pl.col("ex_rights_ref_price").is_not_null()
    & (pl.col("ex_rights_ref_price") > 0)
)
_cap = has_capital_reduction.with_columns(
    pl.when(_ok)
    .then(pl.col("prev_close") / pl.col("ex_rights_ref_price"))
    .otherwise(None)
    .alias("adj_factor"),
    pl.col("ex_rights_ref_price").alias("ref_price"),
)

_bad = _cap.filter(pl.col("adj_factor").is_null())
if _bad.height:
    print(f"capital reduction adj_factor null：{_bad.height} 筆（仍可進停牌對照，但無因子）")
    display(_bad.select(["symbol", "date", "prev_close", "ex_rights_ref_price"]))

cap_red_pairs = _cap.select(
    ["symbol", "date", "adj_factor", "ref_price"]
).with_row_index("row_idx")

joined = cap_red_pairs.join(
    trading_halt_df.select(["symbol", "start_halt", "resume_trading_date"]),
    on="symbol",
    how="inner",
)
in_suspension = joined.filter(
    (pl.col("date") >= pl.col("start_halt"))
    & (
        pl.col("resume_trading_date").is_null()
        | (pl.col("date") < pl.col("resume_trading_date"))
    )
)

# 落在停牌區間的減資：事件日 + 停牌起迄 + 調整因子
capital_reduction_halt_df = (
    in_suspension
    .select([
        "symbol",
        pl.col("date").alias("event_date"),  # 減資／除權基準日（落在停牌內）
        "start_halt",
        "resume_trading_date",
        "adj_factor",
        "ref_price",
    ])
    .unique()
    .sort(["symbol", "event_date", "start_halt"])
)

print(
    f"減資落在停牌區間：{capital_reduction_halt_df.height} 筆；"
    f"有恢復日：{capital_reduction_halt_df.filter(pl.col('resume_trading_date').is_not_null()).height}；"
    f"有 adj_factor：{capital_reduction_halt_df.filter(pl.col('adj_factor').is_not_null()).height}"
)
capital_reduction_halt_df


In [ ]:
# 供 OHLCV join 的減資因子表：
# - 落在停牌區間且有恢復日 → 掛在 resume_trading_date（停牌日 OHLCV 無列）
# - 其餘有 adj 的減資 → 掛在事件日 date
_suspended = (
    capital_reduction_halt_df
    .filter(
        pl.col("resume_trading_date").is_not_null()
        & pl.col("adj_factor").is_not_null()
    )
    .select([
        "symbol",
        pl.col("resume_trading_date").alias("date"),
        "adj_factor",
        "ref_price",
        pl.col("event_date"),
        "start_halt",
        "resume_trading_date",
    ])
)

_in_halt_events = capital_reduction_halt_df.select(
    ["symbol", pl.col("event_date").alias("date")]
).unique()

_not_suspended = (
    _cap.filter(pl.col("adj_factor").is_not_null())
    .join(_in_halt_events, on=["symbol", "date"], how="anti")
    .select([
        "symbol",
        "date",
        "adj_factor",
        "ref_price",
        pl.col("date").alias("event_date"),
        pl.lit(None).cast(pl.Date).alias("start_halt"),
        pl.lit(None).cast(pl.Date).alias("resume_trading_date"),
    ])
)

capital_reduction_only_df = (
    pl.concat([_suspended, _not_suspended], how="diagonal_relaxed")
    .select(["symbol", "date", "adj_factor", "ref_price", "event_date", "start_halt", "resume_trading_date"])
    .sort(["symbol", "date"])
)

print(
    f"capital_reduction_only_df：{_suspended.height} 筆掛恢復日、"
    f"{_not_suspended.height} 筆掛事件日、合計 {capital_reduction_only_df.height}"
)
capital_reduction_only_df.head()



In [ ]:
# 減資 (symbol, date) 必須唯一，否則後續 join 會膨脹列數
_dup = (
    capital_reduction_only_df
    .group_by(["symbol", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .sort(["symbol", "date"])
)
if _dup.height:
    print("capital_reduction_only_df 重複 (symbol, date)：")
    display(_dup)
    display(
        capital_reduction_only_df.join(
            _dup.select(["symbol", "date"]), on=["symbol", "date"], how="inner"
        ).sort(["symbol", "date"])
    )
    raise ValueError(
        f"capital_reduction_only_df 有 {_dup.height} 組重複 (symbol, date)"
    )
print(
    "capital_reduction_only_df（減資）唯一 OK，筆數:",
    capital_reduction_only_df.height,
)
capital_reduction_only_df


#### 面額變更

In [ ]:
# 面額變更：算 adj_factor，再標記落在停牌區間 [start_halt, resume) 的事件日
# （同資本減資 capital_reduction_halt_df 的做法；trading_halt_df 沒有 event_date）
_ok = (
    pl.col("prev_close").is_not_null()
    & (pl.col("prev_close") > 0)
    & pl.col("ex_rights_ref_price").is_not_null()
    & (pl.col("ex_rights_ref_price") > 0)
)
_par = (
    ex_rights_df
    .filter(
        pl.col("par_value_change_shares_thousand").is_not_null()
        & (pl.col("par_value_change_shares_thousand") > 0)
    )
    .with_columns(
        pl.when(_ok)
        .then(pl.col("prev_close") / pl.col("ex_rights_ref_price"))
        .otherwise(None)
        .alias("adj_factor"),
        pl.col("ex_rights_ref_price").alias("ref_price"),
    )
)

_bad = _par.filter(pl.col("adj_factor").is_null())
if _bad.height:
    print(f"par value change adj_factor null：{_bad.height} 筆")
    display(_bad.select(["symbol", "date", "prev_close", "ex_rights_ref_price"]))

par_pairs = _par.select(
    ["symbol", "date", "adj_factor", "ref_price"]
).with_row_index("row_idx")

joined = par_pairs.join(
    trading_halt_df.select(["symbol", "start_halt", "resume_trading_date"]),
    on="symbol",
    how="inner",
)
in_suspension = joined.filter(
    (pl.col("date") >= pl.col("start_halt"))
    & (
        pl.col("resume_trading_date").is_null()
        | (pl.col("date") < pl.col("resume_trading_date"))
    )
)

# 落在停牌區間的面額變更：事件日 + 停牌起迄 + 調整因子
par_value_change_halt_df = (
    in_suspension
    .select([
        "symbol",
        pl.col("date").alias("event_date"),
        "start_halt",
        "resume_trading_date",
        "adj_factor",
        "ref_price",
    ])
    .unique()
    .sort(["symbol", "event_date", "start_halt"])
)

print(
    f"面額變更總筆數：{_par.height}；"
    f"落在停牌區間：{par_value_change_halt_df.height}；"
    f"有恢復日：{par_value_change_halt_df.filter(pl.col('resume_trading_date').is_not_null()).height}；"
    f"有 adj_factor：{par_value_change_halt_df.filter(pl.col('adj_factor').is_not_null()).height}"
)
par_value_change_halt_df


In [ ]:
# 供 OHLCV join 的面額變更因子（同減資）：
# - 落在停牌且有恢復日 → 掛 resume_trading_date
# - 其餘有 adj → 掛事件日
_par_suspended = (
    par_value_change_halt_df
    .filter(
        pl.col("resume_trading_date").is_not_null()
        & pl.col("adj_factor").is_not_null()
    )
    .select([
        "symbol",
        pl.col("resume_trading_date").alias("date"),
        "adj_factor",
        "ref_price",
        pl.col("event_date"),
        "start_halt",
        "resume_trading_date",
    ])
)
_par_in_halt = par_value_change_halt_df.select(
    ["symbol", pl.col("event_date").alias("date")]
).unique()
_par_not_suspended = (
    _par.filter(pl.col("adj_factor").is_not_null())
    .join(_par_in_halt, on=["symbol", "date"], how="anti")
    .select([
        "symbol",
        "date",
        "adj_factor",
        "ref_price",
        pl.col("date").alias("event_date"),
        pl.lit(None).cast(pl.Date).alias("start_halt"),
        pl.lit(None).cast(pl.Date).alias("resume_trading_date"),
    ])
)
par_value_change_only_df = (
    pl.concat([_par_suspended, _par_not_suspended], how="diagonal_relaxed")
    .select(
        ["symbol", "date", "adj_factor", "ref_price", "event_date", "start_halt", "resume_trading_date"]
    )
    .sort(["symbol", "date"])
)
_dup = (
    par_value_change_only_df.group_by(["symbol", "date"]).len().filter(pl.col("len") > 1)
)
if _dup.height:
    display(_dup)
    raise ValueError(f"par_value_change_only_df 有 {_dup.height} 組重複 (symbol, date)")
print(
    f"par_value_change_only_df：掛恢復日 {_par_suspended.height}、"
    f"掛事件日 {_par_not_suspended.height}、合計 {par_value_change_only_df.height}"
)
par_value_change_only_df.head()


## 籌碼數據

### 籌碼動向＆報酬(2008年起)  institutional_return

In [ ]:
import institutional_return
reload_modules([institutional_return])
institutional_return_path = os.path.join(rebuild_root, "institutional_return")
institutional_return_df = read_tej_txt(
    one_txt(institutional_return_path),
    institutional_return.INSTITUTIONAL_RETURN_SCHEMA,
)
institutional_return_df = institutional_return.process_institutional_return(institutional_return_df)

# 只保留 symbol 存在於 OHLCV_df 內的資料
institutional_return_df = institutional_return_df.filter(
    pl.col("symbol").is_in(OHLCV_df["symbol"].unique())
)

institutional_return_df.head()


In [ ]:
# 檢查 institutional_return_df 內所有欄位的 NA 狀況
institutional_return_df.null_count()

### 三大法人買賣超 institutional_net_flow

In [ ]:
import institutional_net_flow
reload_modules([institutional_net_flow])
# 路徑下為子資料夾 OTC / OTC_DE / TSE / TSE_DE，各內一個 .txt
institutional_net_flow_path = os.path.join(rebuild_root, "institutional_net_flow")
institutional_net_flow_raw_df = read_tej_folder(
    institutional_net_flow_path,
    institutional_net_flow.INSTITUTIONAL_NET_FLOW_SCHEMA,
    allow_market_split=True,
)
institutional_net_flow_df = institutional_net_flow.fill_nulls(
    institutional_net_flow_raw_df
).select(institutional_net_flow.keep_col)
institutional_net_flow_df.head()


### 融資融券 Credit_Transactions

In [ ]:
# 處理完NA
import Credit_Transactions
reload_modules([Credit_Transactions])
Credit_Transaction_path = os.path.join(rebuild_root, "Credit_Transactions")

Credit_Transaction_df = read_tej_txt(
        one_txt(Credit_Transaction_path),
        Credit_Transactions.CREDIT_TRANSACTIONS_SCHEMA,
    )
Credit_Transaction_df = Credit_Transactions.process_credit_transactions(Credit_Transaction_df)
#Credit_Transaction_df.head()


### 當沖資料 Day_Trading_Stats

In [ ]:
import Day_Trading_Stats
# day_trade_volume 為NA的資料 同時其餘兩個都是NA
# 原始資料並無缺失數據 所以可以視NA為當天不可當沖
reload_modules([Day_Trading_Stats])
Day_Trading_Stats_path = os.path.join(rebuild_root, "Day_Trading_Stats")
Day_Trading_Stats_raw_df = read_tej_txt(
    one_txt(Day_Trading_Stats_path),
    Day_Trading_Stats.DAY_TRADING_STATS_SCHEMA,
)
Day_Trading_Stats_df = Day_Trading_Stats.process_day_trading_stats(Day_Trading_Stats_raw_df)
# Day_Trading_Stats_raw_df.head()


In [ ]:
Day_Trading_Stats_raw_df.filter(
    pl.col('day_trade_volume').is_null()
)

### 借券餘額 Securities_Lending

有問題 暫時不用

In [ ]:
# import Securities_Lending
# reload_modules([Securities_Lending])
# # lend_broker_balance 沒有NA

# Securities_Lending_path = os.path.join(rebuild_root, "Securities_Lending")
# Securities_Lending_raw_df = read_tej_txt(
#     one_txt(Securities_Lending_path),
#     Securities_Lending.SECURITIES_LENDING_SCHEMA,
# )
# # Securities_Lending_df = Securities_Lending.process_securities_lending(Securities_Lending_raw_df)

# # # 篩選留下ohlcv有的SYMBOL
# # Securities_Lending_df_tmp = Securities_Lending_df.filter(
# #     pl.col('symbol').is_in(OHLCV_df.select('symbol').unique().to_series().implode())
# # )
# # Securities_Lending_raw_df = Securities_Lending_raw_df.select(
# #     Securities_Lending.keep_col
# # )
# # Securities_Lending_raw_df.head()


### 籌碼分佈 share_distribution

In [ ]:
import share_distribution
reload_modules([share_distribution])
share_distribution_path = os.path.join(rebuild_root, "share_distribution")
share_distribution_df = read_tej_folder(
    share_distribution_path,
    share_distribution.SHARE_DISTRIBUTION_SCHEMA,
    allow_market_split=True,
)
share_distribution_df = share_distribution.process_share_distribution(share_distribution_df)
share_distribution_df.head()

## 合併

### 除息 × 配股／減資：同日重疊檢查
分別對 `ex_rights_adj_factor_df`（配股）與 `capital_reduction_only_df`（減資）檢查

In [ ]:
# 檢查：同日是否同時有現金除息 ×（配股 / 減資）因子
cash_meta = (
    dividend_info
    .filter(pl.col("adj_factor").is_not_null())
    .select([
        "symbol",
        "date",
        pl.col("ex_div_ref_price").alias("cash_ref"),
        pl.col("adj_factor").alias("cash_adj"),
    ])
)

def _overlap_check(rights_df: pl.DataFrame, label: str):
    overlap = rights_df.join(cash_meta, on=["symbol", "date"], how="inner")
    same = overlap.filter(pl.col("ref_price") == pl.col("cash_ref"))
    diff = overlap.filter(pl.col("ref_price") != pl.col("cash_ref"))
    print(f"\n[{label}] 同日重疊總筆數: {overlap.height}")
    print(f"  參考價相同: {same.height}")
    print(f"  參考價不同: {diff.height}")
    if same.height:
        print(f"=== {label} 同日且參考價相同（應只保留一側）===")
        display(
            same.select(
                ["symbol", "date", "ref_price", "cash_ref", "adj_factor", "cash_adj"]
            ).sort(["symbol", "date"])
        )
    if diff.height:
        print(f"=== {label} 同日但參考價不同（不可自動相乘）===")
        display(
            diff.select(
                ["symbol", "date", "ref_price", "cash_ref", "adj_factor", "cash_adj"]
            ).sort(["symbol", "date"])
        )
    return same, diff

same_ref, diff_ref = _overlap_check(ex_rights_adj_factor_df, "配股")
same_ref_capred, diff_ref_capred = _overlap_check(
    capital_reduction_only_df, "減資"
)


### 合併配股／減資因子到 OHLCV
- `ex_rights_adj_factor_df`（配股）與 `capital_reduction_only_df`（減資）分開 join
- 同日且參考價相同：丟掉該側，保留現金除息
- 同日但參考價不同：raise


In [ ]:
# 合併：OHLCV（已有現金除息）× 配股 × 減資
# 面額變更改在後面「合併面額變更」格處理
# 同日且參考價相同 → 丟掉該側，保留現金除息
# 同日但參考價不同 → raise
if diff_ref.height or diff_ref_capred.height:
    raise ValueError(
        f"同日參考價不同：配股 {diff_ref.height}、減資 {diff_ref_capred.height}；"
        "請人工決定後再跑"
    )

def _rights_to_join(rights_df: pl.DataFrame, same: pl.DataFrame) -> pl.DataFrame:
    out = (
        rights_df
        .join(same.select(["symbol", "date"]), on=["symbol", "date"], how="anti")
        .select(["symbol", "date", "adj_factor"])
    )
    _dup = out.group_by(["symbol", "date"]).len().filter(pl.col("len") > 1)
    if _dup.height:
        display(_dup)
        raise ValueError(f"合併前仍有重複 (symbol, date)：{_dup.height}")
    return out

ex_rights_to_join = _rights_to_join(ex_rights_adj_factor_df, same_ref)
capred_to_join = _rights_to_join(capital_reduction_only_df, same_ref_capred)

final_df = (
    OHLCV_df
    .join(ex_rights_to_join, on=["symbol", "date"], how="left")
    .with_columns(pl.col("adj_factor_right").fill_null(1).alias("adj_stock"))
    .drop("adj_factor_right")
    .join(capred_to_join, on=["symbol", "date"], how="left")
    .with_columns(pl.col("adj_factor_right").fill_null(1).alias("adj_capred"))
    .drop("adj_factor_right")
    .with_columns(
        (pl.col("adj_factor") * pl.col("adj_stock") * pl.col("adj_capred")).alias(
            "adj_factor"
        )
    )
    .drop(["adj_stock", "adj_capred"])
)

print(
    f"合併完成：丟掉同日同參考價 配股 {same_ref.height} / 減資 {same_ref_capred.height}；"
    f"寫入 配股 {ex_rights_to_join.height} / 減資 {capred_to_join.height}"
)
final_df.head()


In [ ]:
# 可以同時用 Python 3.8+ polars 的 reduce/pipe 簡化多個 join
import functools
import polars as pl

# 欲合併的 dataframe 列表
dfs_to_join = [
    institutional_return_df,
    institutional_net_flow_df ,
    Credit_Transaction_df , 
    Day_Trading_Stats_df  , 
    # Securities_Lending_df , 數據有問題 暫時不用
    share_distribution_df , 
]

final_df = functools.reduce(
    lambda left, right: left.join(right, on=["symbol", "date"], how="left"),
    dfs_to_join,
    final_df
)
final_df.head()

### 減資停牌期間：從 final_df 移除不應存在的交易日
`capital_reduction_halt_df` 區間 `[start_halt, resume)`（含恢復日前一交易日）理論上該 symbol 不應有日線；若有則刪除。

In [ ]:
# 減資停牌 [start_halt, resume)：理論上 OHLCV/final_df 不應有列 → 列出不合理樣本後刪除
_halts = (
    capital_reduction_halt_df
    .filter(pl.col("resume_trading_date").is_not_null())
    .select(["symbol", "start_halt", "resume_trading_date"])
    .unique()
    .sort(["symbol", "start_halt"])
)

# 區間內日曆日
_should_absent = (
    _halts
    .join(calendar_df.select(pl.col("date").alias("cal_date")), how="cross")
    .filter(
        (pl.col("cal_date") >= pl.col("start_halt"))
        & (pl.col("cal_date") < pl.col("resume_trading_date"))
    )
    .select(["symbol", "start_halt", "resume_trading_date", pl.col("cal_date").alias("date")])
)

# 不合理 = 區間內 OHLCV 仍有列的停牌段
_ohlcv_in_halt = (
    OHLCV_df.select(["symbol", "date"])
    .join(
        _should_absent.select(["symbol", "date", "start_halt", "resume_trading_date"]),
        on=["symbol", "date"],
        how="inner",
    )
)
_bad_halts = (
    _ohlcv_in_halt
    .group_by(["symbol", "start_halt", "resume_trading_date"])
    .agg(pl.len().alias("n_ohlcv_in_range"))
    .sort(["symbol", "start_halt"])
)
print(f"區間內仍有 OHLCV 列的停牌段：{_bad_halts.height}（顯示前 5 段）")
#display(_bad_halts.head(5))

_show_halts = _bad_halts.head(5)
for row in _show_halts.iter_rows(named=True):
    sym, st, rs = row["symbol"], row["start_halt"], row["resume_trading_date"]
    print("=" * 60)
    print(f"symbol={sym}  halt=[{st}, {rs})")

    in_range = (
        OHLCV_df
        .filter(
            (pl.col("symbol") == sym)
            & (pl.col("date") >= st)
            & (pl.col("date") < rs)
        )
        .select(["symbol", "date", "close", "volume", "untradable"])
        .sort("date")
    )
    print(f"【刪除前】區間內 OHLCV 共 {in_range.height} 日：")
    display(in_range)
    n_vol0 = in_range.filter(
        pl.col("volume").is_null() | (pl.col("volume") == 0)
    ).height
    print(
        f"  volume==0 或 null：{n_vol0}/{in_range.height}；"
        f"volume>0：{in_range.height - n_vol0}"
    )
    if in_range.height - n_vol0:
        print("  !! 有成交量>0 的列：")
        #display(in_range.filter(pl.col("volume").fill_null(0) > 0))

# 全體被刪列：確認 volume 是否皆為 0
_drop_keys = _should_absent.select(["symbol", "date"]).unique()
_deleted_ohlcv = (
    OHLCV_df
    .join(_drop_keys, on=["symbol", "date"], how="inner")
    .select(["symbol", "date", "close", "volume", "untradable"])
)
_vol_pos = _deleted_ohlcv.filter(pl.col("volume").fill_null(0) > 0)
_vol_zero = _deleted_ohlcv.filter(pl.col("volume").is_null() | (pl.col("volume") == 0))
print("\n=== 全部將刪除列的成交量檢查 ===")
print(f"OHLCV 中會被刪的列：{_deleted_ohlcv.height}")
print(f"  volume==0 或 null：{_vol_zero.height}")
print(f"  volume>0：{_vol_pos.height}")
if _vol_pos.height:
    print("volume>0 樣本（不該在停牌日有量）：")
    display(_vol_pos.sort(["symbol", "date"]).head(20))
else:
    print("ok：被刪日期的成交量皆為 0 或 null")

# 從 final_df 刪除
_before = final_df.height
final_df = final_df.join(_drop_keys, on=["symbol", "date"], how="anti")
print(f"\nfinal_df：{_before} → {final_df.height}（刪 {_before - final_df.height} 列）")

# 同樣前 5 段：刪除後附近還剩哪些日期
print("\n【刪除後】前 5 段附近還剩的日期：")
cals = calendar_df.sort("date")["date"].to_list()
for row in _show_halts.iter_rows(named=True):
    sym, st, rs = row["symbol"], row["start_halt"], row["resume_trading_date"]
    try:
        i_st = cals.index(st)
    except ValueError:
        i_st = next((i for i, d in enumerate(cals) if d >= st), 0)
    try:
        i_rs = cals.index(rs)
    except ValueError:
        i_rs = next((i for i, d in enumerate(cals) if d >= rs), len(cals) - 1)
    win_start = cals[max(0, i_st - 5)]
    win_end = cals[min(len(cals) - 1, i_rs + 5)]

    left = (
        final_df
        .filter(
            (pl.col("symbol") == sym)
            & (pl.col("date") >= win_start)
            & (pl.col("date") <= win_end)
        )
        .select(["symbol", "date", "close", "untradable", "adj_factor"])
        .sort("date")
    )
    still_in = left.filter((pl.col("date") >= st) & (pl.col("date") < rs))
    print("=" * 60)
    print(f"symbol={sym}  halt=[{st}, {rs})  視窗=[{win_start}, {win_end}]")
    print(f"刪除後視窗內剩餘 {left.height} 日；其中仍在停牌區間：{still_in.height}")
    #display(left)


### 合併面額變更

In [ ]:
# 面額變更因子併入 final_df：停牌內事件掛在 resume_trading_date
par_to_join = (
    par_value_change_halt_df
    .filter(
        pl.col("resume_trading_date").is_not_null()
        & pl.col("adj_factor").is_not_null()
    )
    .select([
        "symbol",
        pl.col("resume_trading_date").alias("date"),
        "adj_factor",
        "ref_price",
        "event_date",
        "start_halt",
        "resume_trading_date",
    ])
)

_dup = par_to_join.group_by(["symbol", "date"]).len().filter(pl.col("len") > 1)
if _dup.height:
    display(_dup)
    display(par_to_join.join(_dup.select(["symbol", "date"]), on=["symbol", "date"], how="inner"))
    raise ValueError(f"par_to_join 有 {_dup.height} 組重複 (symbol, date)")

# 與現金除息同日重疊（掛點 = resume 日）
_cash = (
    dividend_info
    .filter(pl.col("adj_factor").is_not_null())
    .select([
        "symbol",
        "date",
        pl.col("ex_div_ref_price").alias("cash_ref"),
        pl.col("adj_factor").alias("cash_adj"),
    ])
)
_overlap = par_to_join.join(_cash, on=["symbol", "date"], how="inner")
_same = _overlap.filter(pl.col("ref_price") == pl.col("cash_ref"))
_diff = _overlap.filter(pl.col("ref_price") != pl.col("cash_ref"))
if _same.height:
    print(f"面額變更與現金除息同日同參考價 {_same.height} 筆 → 丟掉面額側")
    display(_same.select(["symbol", "date", "ref_price", "cash_ref", "adj_factor", "cash_adj"]))
    par_to_join = par_to_join.join(
        _same.select(["symbol", "date"]), on=["symbol", "date"], how="anti"
    )
if _diff.height:
    display(_diff.select(["symbol", "date", "ref_price", "cash_ref", "adj_factor", "cash_adj"]))
    raise ValueError(
        f"面額變更與現金除息同日但參考價不同 {_diff.height} 筆，禁止自動相乘"
    )

_join = par_to_join.select(["symbol", "date", "adj_factor"])
final_df = (
    final_df
    .join(_join, on=["symbol", "date"], how="left")
    .with_columns(pl.col("adj_factor_right").fill_null(1))
    .with_columns(
        (pl.col("adj_factor") * pl.col("adj_factor_right")).alias("adj_factor")
    )
    .drop("adj_factor_right")
)
_hit = (
    final_df
    .join(_join.select(["symbol", "date"]), on=["symbol", "date"], how="semi")
    .select(["symbol", "date", "adj_factor"])
    .sort(["symbol", "date"])
)
print(f"面額變更寫入 {_join.height} 筆（掛恢復日）；final_df 命中 {_hit.height} 列")
_hit.head(10)


In [ ]:
import datetime

# 指定 symbol 和日期區間
my_symbol = "1109"  # 這裡換成你想查詢的 symbol [2015-10-16, 2015-10-30)
start_date = datetime.date(2015, 10, 1)  # 這裡換成你的起始日期
end_date = datetime.date(2015, 10, 16)     # 這裡換成你的結束日期

result = (
    final_df
    .filter(
        (pl.col("symbol") == my_symbol)
        & (pl.col("date") >= pl.lit(start_date))
        & (pl.col("date") <= pl.lit(end_date))
    )
    .sort("date")
).select(['symbol' , 'date' , 'avoid_early'])
print(f"symbol={my_symbol} 區間: [{start_date}, {end_date}] 共 {result.height} 筆")
display(result)

### 合併後檢查

In [ ]:
def fill_margin_net_buy_lot_with_zero(df):
    """
    將 untradable = 0 且 marginable = 1 的資料，將 margin_net_buy_lot 填入 0
    """
    mask = (pl.col("untradable") == 0) & (pl.col("marginable") == 1)
    if "margin_net_buy_lot" in df.columns:
        return df.with_columns(
            pl.when(mask)
            .then(0)
            .otherwise(pl.col("margin_net_buy_lot"))
            .alias("margin_net_buy_lot")
        )
    else:
        print("找不到欄位 'margin_net_buy_lot'，無法填入 0")
        return df

# 使用方法範例
final_df = fill_margin_net_buy_lot_with_zero(final_df)

In [ ]:

# 這邊假設你有一個 ohlcv schema，其 keep_col 屬性為你想要 check 的欄位
# 例如 from your_schema import ohlcv
# keep_cols = ohlcv.keep_col

skip_na_cols = ['open', 'return_day', 'log_return_day', 'spread', 
                'next_limit_up', 'next_limit_down' , 'short_last_cover_date' , 'margin_maintenance_ratio' , 'short_maintenance_ratio' , 
                'account_maintenance_ratio']  # 這些欄位即使在 keep_col 當中也跳過不查


import Corporate_Actions



# Day Trade 的缺失值為不可當沖標的的交易標的造成的 保留NA 


# day trade 暫緩
keep_cols = share_distribution.keep_col

# 僅在 untradable != 1 的狀態下統計
not_untradable_df = final_df.filter(pl.col('untradable') == 0)
filter_df = not_untradable_df


filter_df = filter_df.filter(pl.col('marginable') == 1)

# filter_df = encode_marginable(filter_df)
#filter_df = filter_df.filter(pl.col('marginable') == 0)

all_pass = True

for col in keep_cols:
    if col in skip_na_cols:
        continue
    if col not in filter_df.columns:
        print(f"Column '{col}' 不在 filter_df.columns 中，跳過")
        continue
    n_na = filter_df.filter(pl.col(col).is_null()).height
    total = filter_df.height 
    ratio = n_na / total if total > 0 else 0
    if n_na > 0:
        print(f"Column '{col}': {n_na} NA values ({ratio:.2%})")
        all_pass = False

if all_pass:
    print("所有欄位皆通過檢查")
else:
    print("有欄位未通過檢查")
   

In [ ]:
import duckdb

# 建立一個新的 DuckDB 資料庫於指定路徑
duckdb_path = "/home/newt/quant_project/duckdb/daily_data.duckdb"
con = duckdb.connect(database=duckdb_path, read_only=False)

# print(f"DuckDB 資料庫已建立於: {duckdb_path}")

# # 把 final_df 存進 DuckDB 的 daily_data 表格
# con.execute("CREATE TABLE IF NOT EXISTS daily_data AS SELECT * FROM final_df LIMIT 0")  # 只建表結構
# con.execute("DELETE FROM daily_data")  # 清空 daily_data 表資料（如果已存在）
# con.register('final_df_temp', final_df)
# con.execute("INSERT INTO daily_data SELECT * FROM final_df_temp")
# print("final_df 已成功存入 DuckDB 的 daily_data 表")

In [1]:
# 讀取全部資料
import duckdb

duckdb_path = "/home/newt/quant_project/duckdb/daily_data.duckdb"
con = duckdb.connect(database=duckdb_path, read_only=True)

df = con.execute("SELECT * FROM daily_data").pl()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
df.columns

['symbol',
 'date',
 'open',
 'high',
 'low',
 'close',
 'volume',
 'amount',
 'return_day',
 'turnover',
 'market_value_million',
 'log_return_day',
 'numtrades',
 'limit_status',
 'spread',
 'next_open_ref',
 'next_limit_up',
 'next_limit_down',
 'attention_stock',
 'disposed_stock',
 'full_delivery',
 'market',
 'untradable',
 'avoid_early',
 'adj_factor',
 'foreign_net_buy_lot',
 'it_net_buy_lot',
 'inst_net_buy_lot',
 'margin_net_buy_lot',
 'cash_net_buy_lot',
 'foreign_net_buy_days',
 'it_net_buy_days',
 'dealer_net_buy_days',
 'inst_net_buy_days',
 'margin_net_buy_days',
 'cash_net_buy_days',
 'foreign_streak_return',
 'it_streak_return',
 'dealer_streak_return',
 'inst_streak_return',
 'margin_streak_return',
 'cash_streak_return',
 'foreign_streak_cum_lot',
 'it_streak_cum_lot',
 'dealer_streak_cum_lot',
 'inst_streak_cum_lot',
 'margin_streak_cum_lot',
 'cash_streak_cum_lot',
 'foreign_streak_cum_amt_thousand',
 'it_streak_cum_amt_thousand',
 'dealer_streak_cum_amt_thousand',

## 市場代號/產業/子產業編碼

In [ ]:
import json
import os
import polars as pl

setting_file = '/home/newt/quant_project/setting_files'
exchange_master_map = json.load(open(os.path.join(setting_file, 'exchange_master_map.json')))

# 將 market 字串直接編碼成整數（未列在 map 的值 → null）
df = df.with_columns(
    pl.col("market").replace_strict(exchange_master_map, default=None).cast(pl.Int8)
)


In [ ]:
import Company_Info
reload_modules([Company_Info])
from Company_Info import (
    add_industry_timelines,
    build_rebuild_master_maps,
    encode_industry_columns,
)

company_info = os.path.join(rebuild_root, "company_info")
company_info_df = read_tej_txt(
    one_txt(company_info),
)

# 用本次 TEJ company_info 建新 map（不沿用舊 master_map；不補 M）
industry_master_map, subindustry_master_map = build_rebuild_master_maps(
    company_info_df,
    setting_file,
)
print(
    f"rebuild maps: industry={len(industry_master_map)}, "
    f"subindustry={len(subindustry_master_map)}"
)

df = add_industry_timelines(df, company_info_df)
df = encode_industry_columns(df, industry_master_map, subindustry_master_map)
df.select("symbol", "date", "industry", "subindustry").head()


In [ ]:
import duckdb

# 前面若用 read_only 連線，無法 CREATE；改開可寫連線覆寫 daily_data
duckdb_path = "/home/newt/quant_project/duckdb/daily_data.duckdb"
try:
    con.close()
except Exception:
    pass
con = duckdb.connect(database=duckdb_path, read_only=False)

con.register("df_upd", df)
con.execute("CREATE OR REPLACE TABLE daily_data AS SELECT * FROM df_upd")
con.unregister("df_upd")
print("daily_data rows:", con.execute("SELECT COUNT(*) FROM daily_data").fetchone()[0])


In [ ]:
con.close()